# Task 3 — PostgreSQL Database
## Store & Query Processed Reviews

This notebook:
1. Installs and verifies PostgreSQL is running
2. Creates the `bank_reviews` database
3. Creates the `banks` and `reviews` tables
4. Inserts all analyzed reviews from Task 2
5. Runs verification queries to confirm data integrity


## 1. PostgreSQL Setup on Windows 11

Follow these steps **once** before running any code cells.

---

### Step 1 — Download & Install PostgreSQL

1. Go to: https://www.postgresql.org/download/windows/
2. Click **Download the installer** (EDB installer)
3. Choose the latest version (e.g. PostgreSQL 16)
4. Run the installer with these settings:
   - Installation directory: leave as default
   - Components: keep all checked (PostgreSQL Server, pgAdmin 4, Command Line Tools)
   - Data directory: leave as default
   - **Password**: set a password you will remember (e.g. `yourpassword`) — this is the `postgres` superuser password
   - Port: `5432` (default — do not change)
   - Locale: leave as default
5. Click through to finish. Do NOT install Stack Builder when prompted.

---

### Step 2 — Verify PostgreSQL is running

Open **Services** (press `Win + R`, type `services.msc`, press Enter).
Find `postgresql-x64-16` (or similar). Status should be **Running**.

If it is not running: right-click → Start.

---

### Step 3 — Create the database using pgAdmin 4

pgAdmin 4 was installed alongside PostgreSQL. Use it to create the database:

1. Open **pgAdmin 4** from the Start menu
2. In the left panel, expand: Servers → PostgreSQL 16
3. Enter your password when prompted
4. Right-click **Databases** → **Create** → **Database**
5. Name it: `bank_reviews`
6. Click **Save**

---

### Step 4 — Install psycopg2 in your Python environment

Open your terminal (Anaconda Prompt or CMD) and run:

```bash
pip install psycopg2-binary
```

---

### Step 5 — Update the password below

In Cell 2, change `"yourpassword"` to the password you set during installation.

Once all steps above are done, run the cells in order.


## 2. Imports & Configuration

In [7]:
import sys, os
sys.path.append(os.path.abspath(".."))

import pandas as pd
from src.database import (
    get_connection,
    create_tables,
    insert_banks,
    insert_reviews,
    verify_insertion,
    theme_breakdown,
)

# ── Database credentials ───────────────────────────────────────────────────────
# Change password to match what you set in the terminal above
DB_CONFIG = {
    "dbname":   "bank_reviews",
    "user":     "postgres",
    "password": "admin",   # <-- change this
    "host":     "localhost",
    "port":     5432,
}

# ── Input file from Task 2 ─────────────────────────────────────────────────────
ANALYZED_PATH = "../data/raw/reviews_analyzed.csv"

print("Configuration ready.")

Configuration ready.


## 3. Load Analyzed Reviews

Load the CSV produced by Task 2. This file must have all 8 columns:
`review, rating, date, bank, source, sentiment_label, sentiment_score, identified_theme`

In [8]:
df = pd.read_csv(ANALYZED_PATH)

print(f"Shape   : {df.shape}")
print(f"Columns : {list(df.columns)}")
print(f"\nReviews per bank:")
print(df["bank"].value_counts().to_string())
print(f"\nMissing values:")
print(df.isnull().sum().to_string())
df.head()

Shape   : (1208, 8)
Columns : ['review', 'rating', 'date', 'bank', 'source', 'sentiment_label', 'sentiment_score', 'identified_theme']

Reviews per bank:
bank
Dashen    423
BOA       410
CBE       375

Missing values:
review              0
rating              0
date                0
bank                0
source              0
sentiment_label     0
sentiment_score     0
identified_theme    0


,review,rating,date,bank,source,sentiment_label,sentiment_score,identified_theme
0,Good,5,2026-05-16,CBE,Google Play,positive,0.9998,General
1,🤙🏼🤙🏼,5,2026-05-16,CBE,Google Play,negative,0.6971,General
2,worst,1,2026-05-16,CBE,Google Play,negative,0.9998,General
3,this app very full,5,2026-05-15,CBE,Google Play,positive,0.9974,General
4,good apps,4,2026-05-15,CBE,Google Play,positive,0.9999,General


## 4. Connect to PostgreSQL

If this cell throws an error:
- Make sure PostgreSQL is running: `sudo service postgresql start`
- Make sure the password in DB_CONFIG matches what you set
- Make sure the `bank_reviews` database was created

In [9]:
conn = get_connection(**DB_CONFIG)
print("Connection successful.")

Connected to database: bank_reviews @ localhost:5432
Connection successful.


## 5. Create Tables

Creates two tables:

**`banks`** — one row per bank (parent table)
| Column | Type | Description |
|--------|------|-------------|
| bank_id | SERIAL PRIMARY KEY | Auto-generated unique ID |
| bank_name | VARCHAR UNIQUE | CBE, BOA, Dashen |
| app_id | VARCHAR | Google Play app package name |
| country | VARCHAR | et (Ethiopia) |

**`reviews`** — one row per review (child table)
| Column | Type | Description |
|--------|------|-------------|
| review_id | SERIAL PRIMARY KEY | Auto-generated unique ID |
| bank_id | INT → banks.bank_id | Foreign key linking to parent bank |
| review_text | TEXT | Raw review content |
| rating | INT (1–5) | Star rating with CHECK constraint |
| review_date | DATE | Normalized to YYYY-MM-DD |
| sentiment_label | VARCHAR | positive / negative |
| sentiment_score | FLOAT | DistilBERT confidence (0–1) |
| identified_theme | VARCHAR | Business theme from Task 2 |
| source | VARCHAR | Google Play |
| created_at | TIMESTAMP | Row insertion timestamp |

### Why a foreign key?
Storing `bank_id` (an integer) in every review row is far more efficient
than storing the full bank name string. It also enforces referential integrity —
you cannot insert a review for a bank that does not exist in the banks table.

In [10]:
create_tables(conn)

Tables created (or already exist): banks, reviews


## 6. Insert Banks

Insert the three banks into the `banks` table.
`ON CONFLICT DO NOTHING` makes this safe to re-run.

In [11]:
bank_ids = insert_banks(conn)
print(f"Bank ID mapping: {bank_ids}")

Banks inserted/verified: {'CBE': 1, 'BOA': 2, 'Dashen': 3}
Bank ID mapping: {'CBE': 1, 'BOA': 2, 'Dashen': 3}


## 7. Insert Reviews

Bulk-insert all reviews using `execute_values` — this sends all rows
in a single SQL statement instead of one per row, making it ~50x faster.

In [12]:
inserted = insert_reviews(conn, df, bank_ids)
print(f"\nTotal rows inserted: {inserted}")

Inserted : 1208 reviews

Total rows inserted: 1208


## 8. Verify Insertion

Run SQL queries to confirm the data loaded correctly.
Compare these numbers against your Task 1 and Task 2 outputs.

In [13]:
summary = verify_insertion(conn)
print("=== Summary per bank ===")
print(summary.to_string(index=False))

=== Summary per bank ===
bank_name  review_count  avg_rating  pct_positive earliest_date latest_date
   Dashen           423        3.79          57.9    2025-08-14  2026-05-16
      BOA           410        3.33          44.1    2025-02-23  2026-05-16
      CBE           375        3.92          57.1    2026-03-04  2026-05-16


c:\users\hp\documents\10x\fintech-review-analytics\src\database.py:220: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


In [14]:
themes = theme_breakdown(conn)
print("=== Theme breakdown per bank ===")
print(themes.to_string(index=False))

=== Theme breakdown per bank ===
bank_name        identified_theme  review_count
      BOA                 General           274
      BOA Transaction Performance            45
      BOA           App Stability            41
      BOA          Account Access            16
      BOA        Customer Support            13
      BOA        Feature Requests            13
      BOA  Balance & Account Info             4
      BOA             UI & Design             4
      CBE                 General           261
      CBE Transaction Performance            51
      CBE           App Stability            31
      CBE        Customer Support            18
      CBE        Feature Requests             9
      CBE             UI & Design             2
      CBE          Account Access             2
      CBE  Balance & Account Info             1
   Dashen                 General           277
   Dashen Transaction Performance            62
   Dashen           App Stability            26
   Dash

c:\users\hp\documents\10x\fintech-review-analytics\src\database.py:237: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


## 9. Explore with Raw SQL

Run SQL queries directly to answer business questions.
These are the kinds of queries you will reference in your Task 4 report.

In [15]:
import pandas as pd

# Query 1: Average rating per bank
q1 = pd.read_sql("""
    SELECT b.bank_name,
           ROUND(AVG(r.rating)::NUMERIC, 2) AS avg_rating,
           COUNT(*) AS total_reviews
    FROM reviews r
    JOIN banks b ON r.bank_id = b.bank_id
    GROUP BY b.bank_name
    ORDER BY avg_rating DESC;
""", conn)

print("Q1 — Average rating per bank:")
print(q1.to_string(index=False))

Q1 — Average rating per bank:
bank_name  avg_rating  total_reviews
      CBE        3.92            375
   Dashen        3.79            423
      BOA        3.33            410


C:\Users\hp\AppData\Local\Temp\ipykernel_3640\1208475929.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q1 = pd.read_sql("""


In [16]:
# Query 2: Most common themes for negative reviews only
q2 = pd.read_sql("""
    SELECT b.bank_name,
           r.identified_theme,
           COUNT(*) AS negative_count
    FROM reviews r
    JOIN banks b ON r.bank_id = b.bank_id
    WHERE r.sentiment_label = 'negative'
    GROUP BY b.bank_name, r.identified_theme
    ORDER BY b.bank_name, negative_count DESC;
""", conn)

print("Q2 — Top negative themes per bank:")
print(q2.to_string(index=False))

Q2 — Top negative themes per bank:
bank_name        identified_theme  negative_count
      BOA                 General             120
      BOA           App Stability              40
      BOA Transaction Performance              32
      BOA          Account Access              14
      BOA        Feature Requests              12
      BOA        Customer Support               4
      BOA             UI & Design               4
      BOA  Balance & Account Info               3
      CBE                 General              88
      CBE Transaction Performance              31
      CBE           App Stability              29
      CBE        Feature Requests               6
      CBE        Customer Support               4
      CBE  Balance & Account Info               1
      CBE          Account Access               1
      CBE             UI & Design               1
   Dashen                 General              93
   Dashen Transaction Performance              32
   Dashen      

C:\Users\hp\AppData\Local\Temp\ipykernel_3640\3127826474.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q2 = pd.read_sql("""


In [17]:
# Query 3: Monthly review trend per bank
q3 = pd.read_sql("""
    SELECT b.bank_name,
           TO_CHAR(r.review_date, 'YYYY-MM') AS month,
           COUNT(*) AS review_count,
           ROUND(AVG(r.rating)::NUMERIC, 2) AS avg_rating
    FROM reviews r
    JOIN banks b ON r.bank_id = b.bank_id
    GROUP BY b.bank_name, month
    ORDER BY b.bank_name, month;
""", conn)

print("Q3 — Monthly trend:")
print(q3.to_string(index=False))

Q3 — Monthly trend:
bank_name   month  review_count  avg_rating
      BOA 2025-02             2        3.00
      BOA 2025-03            33        2.61
      BOA 2025-04            15        2.73
      BOA 2025-05            14        2.57
      BOA 2025-06            20        3.15
      BOA 2025-07            23        3.74
      BOA 2025-08            19        3.53
      BOA 2025-09            19        3.84
      BOA 2025-10            29        3.10
      BOA 2025-11            29        2.69
      BOA 2025-12            27        3.48
      BOA 2026-01            27        2.44
      BOA 2026-02            14        3.00
      BOA 2026-03            90        3.92
      BOA 2026-04            37        3.62
      BOA 2026-05            12        4.08
      CBE 2026-03           107        3.85
      CBE 2026-04           197        3.93
      CBE 2026-05            71        4.00
   Dashen 2025-08            29        3.76
   Dashen 2025-09            55        3.45
   Dashen 20

C:\Users\hp\AppData\Local\Temp\ipykernel_3640\1574386113.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q3 = pd.read_sql("""


In [18]:
# Query 4: 1-star reviews only — what are users most angry about?
q4 = pd.read_sql("""
    SELECT b.bank_name,
           r.identified_theme,
           COUNT(*) AS one_star_count
    FROM reviews r
    JOIN banks b ON r.bank_id = b.bank_id
    WHERE r.rating = 1
    GROUP BY b.bank_name, r.identified_theme
    ORDER BY b.bank_name, one_star_count DESC;
""", conn)

print("Q4 — 1-star review themes:")
print(q4.to_string(index=False))

Q4 — 1-star review themes:
bank_name        identified_theme  one_star_count
      BOA                 General              69
      BOA           App Stability              31
      BOA Transaction Performance              17
      BOA          Account Access              10
      BOA        Feature Requests               9
      BOA        Customer Support               3
      BOA  Balance & Account Info               3
      BOA             UI & Design               3
      CBE                 General              27
      CBE           App Stability              17
      CBE Transaction Performance              17
      CBE        Customer Support               4
      CBE        Feature Requests               3
   Dashen                 General              50
   Dashen Transaction Performance              14
   Dashen           App Stability              12
   Dashen        Feature Requests               8
   Dashen             UI & Design               3
   Dashen  Balance & Ac

C:\Users\hp\AppData\Local\Temp\ipykernel_3640\2344852876.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q4 = pd.read_sql("""


## 10. Close Connection

Always close the connection when you are done.

In [19]:
conn.close()
print("Connection closed.")

Connection closed.


## 11. Schema Reference

```
banks
─────────────────────────────────
bank_id    SERIAL PRIMARY KEY
bank_name  VARCHAR(100) UNIQUE
app_id     VARCHAR(200)
country    VARCHAR(10)

        │  (one bank → many reviews)
        │
        ▼

reviews
─────────────────────────────────
review_id        SERIAL PRIMARY KEY
bank_id          INT → banks.bank_id   (FK)
review_text      TEXT
rating           INT  CHECK (1–5)
review_date      DATE
sentiment_label  VARCHAR(20)
sentiment_score  FLOAT
identified_theme VARCHAR(100)
source           VARCHAR(50)
created_at       TIMESTAMP
```
